In [1]:
#imports
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T

from gammanet.models import VGG16GammaNetV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
import cv2
import numpy as np

def create_line_image(size=256):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    cv2.line(img, (20, size//2), (size-20, size//2), (255,255,255), 2)
    return img

def create_curve_image(size=256):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    center = (size//2, size//2)
    cv2.ellipse(img, center, (80,80), 0, 0, 180, (255,255,255), 2)
    return img

In [3]:
#Paths
input_dir = "/home/yentl/pytorch_gammanet/Images"
output_dir = "/home/yentl/pytorch_gammanet/Output_Images"
checkpoint_path = "/home/yentl/pytorch_gammanet/checkpoint_epoch_40.pt"

os.makedirs(output_dir, exist_ok=True)

In [19]:
#Inspect weight dimensions
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

config = checkpoint['config']['model']

model = VGG16GammaNetV2(config)
model.load_state_dict(checkpoint['model_state_dict'], strict=False)

model.to(device)
model.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Validation F1: {checkpoint.get('best_metric', 'N/A')}")

Loaded model from epoch 40
Validation F1: 0.6260475346234085


In [5]:
#Overview
print(model)

VGG16GammaNetV2(
  (block1_conv): VGG16Block(
    (layers): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block2_conv): VGG16Block(
    (layers): Sequential(
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
    )
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block3_conv): VGG16Block(
    (layers): Sequential(
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     

In [6]:
#Dimensions
print("\n=== fGRU Weight Shapes ===")

for i in range(5):
    fgru = getattr(model, f"fgru_{i}")

    print(f"\nfGRU_{i}")
    print("W_exc:", list(fgru.W_exc.shape))
    print("W_inh:", list(fgru.W_inh.shape))


=== fGRU Weight Shapes ===

fGRU_0
W_exc: [64, 64, 3, 3]
W_inh: [64, 64, 3, 3]

fGRU_1
W_exc: [128, 128, 3, 3]
W_inh: [128, 128, 3, 3]

fGRU_2
W_exc: [256, 256, 3, 3]
W_inh: [256, 256, 3, 3]

fGRU_3
W_exc: [512, 512, 3, 3]
W_inh: [512, 512, 3, 3]

fGRU_4
W_exc: [512, 512, 3, 3]
W_inh: [512, 512, 3, 3]


## Meaning of dimensions

Example:

W_exc shape = [C, C, 3, 3]

This means:
- C = number of feature channels (e.g., 64, 128, 256)
- 3x3 = spatial neighborhood

Interpretation:

Each feature channel has:
→ connections to ALL other channels
→ AND spatial neighbors

So:

Each E neuron:
- is located at (x, y, channel)
- connects to:
    - nearby pixels (3x3)
    - all feature channels

This means E/I are NOT just per-pixel scalars.
They are spatial + feature-aware units.

In [7]:
#Hooks (Conv + fGRU)
activations = {}
ei_states = {}

def get_activation(name):
    def hook(module, input, output):
        if isinstance(output, tuple):
            output = output[0]
        activations[name] = output.detach()
    return hook

def get_fgru_state(name):
    def hook(module, input, output):
        if hasattr(module, "excitation"):
            ei_states[name+"_E"] = module.excitation.detach()
        if hasattr(module, "inhibition"):
            ei_states[name+"_I"] = module.inhibition.detach()
    return hook

# VGG layers
vgg_blocks = {
    "block1": model.block1_conv,
    "block2": model.block2_conv,
    "block3": model.block3_conv,
    "block4": model.block4_conv,
    "block5": model.block5_conv,
}

#Conv layers
for block_name, block in vgg_blocks.items():
    for i, layer in enumerate(block.layers):
        layer.register_forward_hook(get_activation(f"{block_name}_layer{i}"))

# fGRU layers
for i in range(5):
    getattr(model, f"fgru_{i}").register_forward_hook(
        get_activation(f"fgru_{i}")
    )

# E/I hooks
for i in range(5):
    getattr(model, f"fgru_{i}").register_forward_hook(
        get_fgru_state(f"fgru_{i}")
    )

In [8]:
#Image Transform
transform = T.Compose([
    T.Resize((256,256)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [9]:
#Feature map visualization
def save_feature_maps(tensor, layer_name, output_folder, num_channels=6):

    fmap = tensor.squeeze(0)
    n = min(num_channels, fmap.shape[0])

    plt.figure(figsize=(12,4))

    for i in range(n):
        plt.subplot(1,n,i+1)
        plt.imshow(fmap[i].cpu(), cmap="viridis")
        plt.axis("off")

    plt.suptitle(f"{layer_name} (first {n} channels)")
    plt.tight_layout()

    plt.savefig(os.path.join(output_folder, f"{layer_name}.png"))
    plt.close()

In [10]:
def compute_stats(tensor):
    return {
        "mean": tensor.mean().item(),
        "std": tensor.std().item(),
        "min": tensor.min().item(),
        "max": tensor.max().item()
    }

In [11]:
def analyze_shape_response(img, label):

    image = Image.fromarray(img)
    input_tensor = transform(image).unsqueeze(0).to(device)

    activations.clear()

    with torch.no_grad():
        _ = model(input_tensor)

    print(f"\n=== {label} ===")

    for name, fmap in activations.items():
        fmap = fmap.squeeze(0)
        channel_means = fmap.view(fmap.shape[0], -1).mean(dim=1)
        topk = torch.topk(channel_means, k=5)

        print(f"{name} → top channels: {topk.indices.tolist()}")


In [12]:
def analyze_ei_balance():

    print("\n=== E/I BALANCE ===")

    for key in ei_states:

        tensor = ei_states[key]

        mean = tensor.mean().item()
        std = tensor.std().item()

        print(f"{key}: mean={mean:.4f}, std={std:.4f}")

In [17]:
def disable_topdown(model, level):

    for i in range(level+1, 5):
        fgru = getattr(model, f"fgru_{i}")

        def forward_no_topdown(x, h_exc, h_inh, *args, **kwargs):

            if h_exc is None:
                h_exc = torch.zeros_like(x)

            if h_inh is None:
                h_inh = torch.zeros_like(x)

            return h_exc, h_inh, None

        fgru.forward = forward_no_topdown

    print(f"Top-down disabled above level {level}")

In [14]:
def dataset_summary(input_dir):

    image_files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith((".png",".jpg",".jpeg",".bmp",".tiff"))
    ]

    print("\n=== DATASET SUMMARY ===")
    print("Total images:", len(image_files))

In [20]:
# Dataset info
dataset_summary(input_dir)

# Shape selectivity
line_img = create_line_image()
curve_img = create_curve_image()

analyze_shape_response(line_img, "STRAIGHT LINE")
analyze_shape_response(curve_img, "CURVE")

# Top-down test (use one image)
test_image = Image.open(os.path.join(input_dir, os.listdir(input_dir)[0])).convert("RGB")
input_tensor = transform(test_image).unsqueeze(0).to(device)

with torch.no_grad():
    out_full = model(input_tensor)

disable_topdown(model, level=2)

with torch.no_grad():
    out_no_td = model(input_tensor)

print("Top-down difference:", torch.abs(out_full - out_no_td).mean().item())


=== DATASET SUMMARY ===
Total images: 2

=== STRAIGHT LINE ===

=== CURVE ===
Top-down disabled above level 2
Top-down difference: nan


In [21]:
for image_name in os.listdir(input_dir):

    if not image_name.lower().endswith((".png",".jpg",".jpeg",".bmp",".tiff")):
        continue

    print(f"\nProcessing {image_name}")

    base_name = os.path.splitext(image_name)[0]

    image_output_dir = os.path.join(output_dir, base_name)
    featuremap_dir = os.path.join(image_output_dir, "featuremaps")

    os.makedirs(featuremap_dir, exist_ok=True)

    # Load image
    image = Image.open(os.path.join(input_dir, image_name)).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    activations.clear()
    ei_states.clear()


    # Forward
    with torch.no_grad():
        output = model(input_tensor)
        edge_prob = torch.sigmoid(output)

    analyze_ei_balance()    
    edge_map = edge_prob[0,0].cpu().numpy()

    # -----------------------------
    # EDGE STATISTICS
    # -----------------------------
    print("\nEdge statistics:")
    print("Mean:", edge_map.mean())
    print("Std:", edge_map.std())
    print("Max:", edge_map.max())

    # -----------------------------
    # THRESHOLD ANALYSIS
    # -----------------------------
    thresholds = np.arange(0.1, 1.1, 0.1)
    edge_ratios = []

    fig, axes = plt.subplots(2,5, figsize=(15,6))

    for idx, t in enumerate(thresholds):
        binary = edge_map > t
        ratio = binary.sum() / binary.size
        edge_ratios.append(ratio)

        axes[idx//5, idx%5].imshow(binary, cmap="gray")
        axes[idx//5, idx%5].set_title(f"t={t:.1f}\nr={ratio:.2f}")
        axes[idx//5, idx%5].axis("off")

    plt.savefig(os.path.join(image_output_dir, "threshold_overview.png"))
    plt.close()

    # Curve
    plt.figure()
    plt.plot(thresholds, edge_ratios, marker="o")
    plt.title("Edge Ratio Curve")
    plt.savefig(os.path.join(image_output_dir, "threshold_curve.png"))
    plt.close()

    # Overview
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(image)
    plt.title("Input")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(edge_map, cmap="gray")
    plt.title("Edge map")
    plt.axis("off")

    plt.savefig(os.path.join(image_output_dir, f"edges_overview_{base_name}.png"))
    plt.close()

    # -----------------------------
    # FEATURE MAPS + STATS
    # -----------------------------
    print("\nLayer activity:")

    for name, fmap in activations.items():

        print(f"{name}: {list(fmap.shape)} | stats: {compute_stats(fmap)}")

        save_feature_maps(fmap, name, featuremap_dir)


Processing straight_low_BL_0_J000_013.png

=== E/I BALANCE ===

Edge statistics:
Mean: nan
Std: nan
Max: nan

Layer activity:

Processing straight_high_BL_0_J000_013.png

=== E/I BALANCE ===

Edge statistics:
Mean: nan
Std: nan
Max: nan

Layer activity:


## Meaning of shapes?

Example:

Exc shape: [1, 64, 256, 256]

This means:

- 1 = batch
- 64 = number of E neurons per pixel
- 256x256 = spatial map

So:

At each pixel:
→ you have 64 excitatory neurons

Same for inhibitory.

So total E neurons:
64 × 256 × 256 ≈ 4 million neurons

These neurons are NOT independent:
They are connected via convolution:

→ spatial neighbors interact (3x3)
→ feature channels interact

This is how contours can "link" across space.